# Optimal policy evaluation using Temporal Differences Learning

Consider the MDP carachterized by $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$.

We consider the common case where the probabilities of transition are unknown, i.e. $\mathcal{P}:=\{P^{a}:\, a\in\mathcal{A}\}$ is unknown.

>A common approach to approximate the optimal policy consists in alternating the following two steps:
>1) updating the value function by the temporal differences rule TD(0);
>2) updating the policy by chosing the action that maximizes the value function.

**The TD(0) estimation of the value function**:

The value function is updated as follows:

$$
v_{\pi}(s_{n}) \leftarrow v_{\pi}(s_{n}) + \eta (R_{n+1} + \gamma * v_{\pi}(s_{n+1}) − v_{\pi}(s_{n})),
$$

where $\eta > 0$ is the increment.

**The TD(n) estimation of the value function**:

Under the assumptions of a stationary policy, consider the value function

$$
v_{\pi,n}(s):=\mathbb{E}_{\pi} \left[ G_{n} \mid S_{n}=s \right],
$$

where $G_{n}:=\sum\limits_{k=0}^{T \wedge \tau} \gamma^{k} R_{1+k+n} $ is the *objective function* at time $0$, with $R_{1+k+n}$ the reward at time $1+k+n$, $T$ a finite horizon and $\tau$ the minimum time to reach an absorbing state. The objective function selects a random trajectory starting from state $s$ under the policy $\pi$.

The value function is updated as follows:

$$
v_{\pi}(s_{n}) \leftarrow v_{\pi}(s_{n}) + \eta (G_{n} − v_{\pi}(s_{n})),
$$

where $\eta > 0$ is the increment.

In [13]:
import numpy as  np

In [14]:
def TD0(P_trans_allow, Pi, reward, N_max, eta, v_0, s_0, gamma, absorbing_state=None):
    '''
    Estimates the value function v_pi using TD(0).

    P_trans_allow[a, s, s'] : 1 if the transition s -> s' is allowed under action a
                              (the true transition probabilities are unknown)
    Pi[s]         : action chosen by the deterministic policy in state s
    reward[a, s, s']   : reward matrix
    N_max         : number of updates
    eta           : learning rate (step size)
    v_0           : initial values of v
    s_0           : initial state
    gamma         : discount factor
    absorbing_state : list of absorbing states known by the expert
    '''

    if absorbing_state is None:          # pas d'etat absorbant fourni : liste vide
        absorbing_state = []

    v = np.array(v_0, dtype=float)
    n_states = reward.shape[1]

    non_absorbing = [s for s in range(n_states) if s not in absorbing_state]

    s = s_0

    for _ in range(N_max):
        # pi(a|s)=1
        a = Pi[s]

        # Nouvel etat : tirage uniforme parmi les etats autorises
        possible_next_states = np.flatnonzero(P_trans_allow[a, s] > 0)
        if possible_next_states.size == 0:
            raise ValueError(
                f"No allowed next state for state {s}, action {a}. "
                "Each row of P_trans_allow[a, s, :] must contain at least one non-zero entry."
            )
        
        s_prime = np.random.choice(possible_next_states)

        # Mise a jour TD(0)
        v[s] = v[s] + eta * (reward[a, s, s_prime] + gamma * v[s_prime] - v[s])

        # Si s_prime est absorbant : on repart d'un etat non absorbant choisi
        # uniformement ; sinon on continue depuis S_prime
        if s_prime in absorbing_state:
            s = np.random.choice(non_absorbing)
        else:
            s = s_prime

    return v

### Example 1:

Small example from the course.

In [15]:
N_max = 10000      # number of trajectories

# states = [high, low]
# actions = [s, w, c]

R_s = np.array([[5, 5], [-10, 5]])
R_w = np.array([[1, 0], [0, 1]])
R_c = np.array([[0, 0], [0, 0]])
R_complet = np.stack([R_s, R_w, R_c], axis=0) 
# matrix of rewards

P_s_allowed = np.array([[1, 1], [1, 1]])          
P_w_allowed = np.array([[1, 0], [0, 1]])
P_c_allowed = np.array([[1 ,0], [1 ,0]])
P_act_allowed = np.stack([P_s_allowed, P_w_allowed, P_c_allowed], axis=0)
# environmental matrix P[a,s,s'] with 1 where the transition is eventually allowed

pi_s_w = [0, 1]
pi_s_s = [0, 0]
pi_s_c = [0, 2]
# 3 deterministic policy on states

gamma = 0.8

eta_1 = 0.5
eta_2 = 0.05
eta_3 = 0.01

v_0 = [50,50]

s_0 = 0

print("Politique s/w, eta 1",TD0(P_act_allowed, pi_s_w, R_complet, N_max, eta_1, v_0, s_0, gamma))
print("Politique s/s, eta 1",TD0(P_act_allowed, pi_s_s, R_complet, N_max, eta_1, v_0, s_0, gamma))
print( "Politque s/c, eta 1",TD0(P_act_allowed, pi_s_c, R_complet, N_max, eta_1, v_0, s_0, gamma))

print("Politique s/w, eta 2",TD0(P_act_allowed, pi_s_w, R_complet, N_max, eta_2, v_0, s_0, gamma))
print("Politique s/s, eta 2",TD0(P_act_allowed, pi_s_s, R_complet, N_max, eta_2, v_0, s_0, gamma))
print( "Politque s/c, eta 2",TD0(P_act_allowed, pi_s_c, R_complet, N_max, eta_2, v_0, s_0, gamma))

print("Politique s/w, eta 3",TD0(P_act_allowed, pi_s_w, R_complet, N_max, eta_3, v_0, s_0, gamma))
print("Politique s/s, eta 3",TD0(P_act_allowed, pi_s_s, R_complet, N_max, eta_3, v_0, s_0, gamma))
print( "Politque s/c, eta 3",TD0(P_act_allowed, pi_s_c, R_complet, N_max, eta_3, v_0, s_0, gamma))

Politique s/w, eta 1 [47.5  5. ]
Politique s/s, eta 1 [ 5.64182354 -7.76879928]
Politque s/c, eta 1 [15.54681105 12.39887069]
Politique s/w, eta 2 [49.75  5.  ]
Politique s/s, eta 2 [9.78635113 1.75810564]
Politque s/c, eta 2 [17.46051554 13.88342706]
Politique s/w, eta 3 [49.95        5.00000009]
Politique s/s, eta 3 [9.39865548 1.71692884]
Politque s/c, eta 3 [17.80816049 14.23561671]


### Example 2:

Small example from the course.

In [16]:
# allowed trajectories
P_act_allow_s = np.array([
    [1, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 1, 0, 0],
    [0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1]])

P_act_allow_w = np.array([
    [1, 0, 1, 0, 0, 0, 0],
    [1, 0, 1, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1],   
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1]])

P_act_allowed_etudiant = np.stack([P_act_allow_s, P_act_allow_w], axis=0)

# rewards
R_allow_s = np.array([
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, -10, 0, 0],
    [0, 1, -1, 0, 0, 0, 0],
    [0, 0, 0, -10, 0, 100, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0]])

R_allow_w = np.array([
    [0, 0, -1, 0, 0, 0, 0],
    [0, 0, -1, 0, 0, 0, 0],
    [0, 0, -1, -10, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, -1000],   
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0]])
R_allowed_etudiant = np.stack([R_allow_s, R_allow_w], axis=0)

absorbing_state = [4, 5, 6]
pi_SWWS = [0, 1, 1, 0, 0, 0, 0]   # Repose, Travaille, Travaille, Repose
v_0_etudiant = [0] * 7
s_0 = 3
N_max = 100000
gamma = 0.8


print(TD0(P_act_allowed_etudiant, pi_SWWS, R_allowed_etudiant, N_max, 0.01, v_0_etudiant, s_0, gamma, absorbing_state))
print(TD0(P_act_allowed_etudiant, pi_SWWS, R_allowed_etudiant, N_max, 0.05, v_0_etudiant, s_0, gamma, absorbing_state))
print(TD0(P_act_allowed_etudiant, pi_SWWS, R_allowed_etudiant, N_max, 0.1, v_0_etudiant, s_0, gamma, absorbing_state))

[15.9310206  22.79588119 42.2284059  75.89370771  0.          0.
  0.        ]
[17.86571978 24.05209029 42.6176834  77.32015185  0.          0.
  0.        ]
[17.33359573 22.85378513 43.58460205 78.89065886  0.          0.
  0.        ]


## Assumptions: $P_{s,s'}^{a}$ known

To compare the results of the example, here we assume that the probabilities of transition are known. The new state s' is chosen under the matrix of transition probabilities $P_{s,s'}^{a}$, instaed than randomly uniformly.

In [17]:
import numpy as np

In [18]:
def TD0(P_act, Pi, reward, N_max, eta, v_0, s_0, gamma, absorbing_state=None):
    '''
    Estimates the value function v_pi using TD(0).

    P_act[a, s, s'] : matrix of transitions P_{s,s'}^{a}
    Pi[s]         : action chosen by the deterministic policy in state s
    reward[a, s, s']   : reward matrix
    N_max         : number of updates
    eta           : learning rate (step size)
    v_0           : initial values of v
    s_0           : initial state
    gamma         : discount factor
    absorbing_state : list of absorbing states known by the expert
    '''

    if absorbing_state is None:          # pas d'etat absorbant fourni : liste vide
        absorbing_state = []

    v = np.array(v_0, dtype=float)
    n_states = reward.shape[1]

    non_absorbing = [s for s in range(n_states) if s not in absorbing_state]

    s = s_0

    for _ in range(N_max):
        # pi(a|s)=1
        a = Pi[s]

        # Choix nouveau etat selon les probabilitées de transition
        s_prime = np.random.choice(n_states, p=P_act[a, s])

        # Mise a jour TD(0)
        v[s] = v[s] + eta * (reward[a, s, s_prime] + gamma * v[s_prime] - v[s])

        # Si s_prime est absorbant : on repart d'un etat non absorbant choisi
        # uniformement ; sinon on continue depuis S_prime
        if s_prime in absorbing_state:
            s = np.random.choice(non_absorbing)
        else:
            s = s_prime

    return v

### Example 1:

In [19]:
N_max = 10000      # number of trajectories

# states = [high, low]
# actions = [s, w, c]

R_s = np.array([[5, 5], [-10, 5]])
R_w = np.array([[1, 0], [0, 1]])
R_c = np.array([[0, 0], [0, 0]])
R_complet = np.stack([R_s, R_w, R_c], axis=0) 
# matrix of rewards

P_s = np.array([[0.3,0.7],[0.4,0.6]])
P_w = np.eye(2)
P_c = np.array([[1,0],[1,0]])
P_act = np.stack([P_s,P_w,P_c], axis=0)
# P[a][s][s'] matrix of transition probabilites

pi_s_w = [0, 1]
pi_s_s = [0, 0]
pi_s_c = [0, 2]
# 3 deterministic policy on states

gamma = 0.8

eta_1 = 0.5
eta_2 = 0.05
eta_3 = 0.01

v_0 = [20,20]

s_0 = 0

print("Politique s/w, eta 1",TD0(P_act, pi_s_w, R_complet, N_max, eta_1, v_0, s_0, gamma))
print("Politique s/s, eta 1",TD0(P_act, pi_s_s, R_complet, N_max, eta_1, v_0, s_0, gamma))
print( "Politque s/c, eta 1",TD0(P_act, pi_s_c, R_complet, N_max, eta_1, v_0, s_0, gamma))

print("Politique s/w, eta 2",TD0(P_act, pi_s_w, R_complet, N_max, eta_2, v_0, s_0, gamma))
print("Politique s/s, eta 2",TD0(P_act, pi_s_s, R_complet, N_max, eta_2, v_0, s_0, gamma))
print( "Politque s/c, eta 2",TD0(P_act, pi_s_c, R_complet, N_max, eta_2, v_0, s_0, gamma))

print("Politique s/w, eta 3",TD0(P_act, pi_s_w, R_complet, N_max, eta_3, v_0, s_0, gamma))
print("Politique s/s, eta 3",TD0(P_act, pi_s_s, R_complet, N_max, eta_3, v_0, s_0, gamma))
print( "Politque s/c, eta 3",TD0(P_act, pi_s_c, R_complet, N_max, eta_3, v_0, s_0, gamma))

Politique s/w, eta 1 [20.5  5. ]
Politique s/s, eta 1 [ 2.02307415 -7.15910246]
Politque s/c, eta 1 [14.47633849 11.65972413]
Politique s/w, eta 2 [20.05  5.  ]
Politique s/s, eta 2 [10.42792088  4.03631303]
Politque s/c, eta 2 [15.65160635 12.65297354]
Politique s/w, eta 3 [20.01        5.00000003]
Politique s/s, eta 3 [9.84458529 5.31559201]
Politque s/c, eta 3 [15.91732636 12.73528864]


### Example 2:

In [21]:
# allowed trajectories
#transition matrices for each action
P_0ex = np.array([[0.5, 0.5, 0, 0, 0, 0, 0],
                [0, 0.6, 0, 0, 0.4, 0, 0],
                [0, 0.4, 0.6, 0, 0, 0, 0],
                [0, 0, 0, 0.1, 0, 0.9, 0],
                [0, 0, 0, 0, 1, 0, 0],
                [0, 0, 0, 0, 0, 1, 0],
                [0, 0, 0, 0, 0, 0, 1]])

P_1ex = np.array([[0.5, 0, 0.5, 0, 0, 0, 0],
                [0.3, 0, 0.7, 0, 0, 0, 0],
                [0, 0, 0.5, 0.5, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 1],
                [0, 0, 0, 0, 1, 0, 0],
                [0, 0, 0, 0, 0, 1, 0],
                [0, 0, 0, 0, 0, 0, 1]])

P_act = np.stack([P_0ex, P_1ex], axis=0)

# rewards
R_allow_s = np.array([
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, -10, 0, 0],
    [0, 1, -1, 0, 0, 0, 0],
    [0, 0, 0, -10, 0, 100, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0]])

R_allow_w = np.array([
    [0, 0, -1, 0, 0, 0, 0],
    [0, 0, -1, 0, 0, 0, 0],
    [0, 0, -1, -10, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, -1000],   
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0]])
R_allowed_etudiant = np.stack([R_allow_s, R_allow_w], axis=0)

absorbing_state = [4, 5, 6]
pi_SWWS = [0, 1, 1, 0, 0, 0, 0]   # Repose, Travaille, Travaille, Repose
v_0_etudiant = [0] * 7
s_0 = 3
N_max = 100000
gamma = 0.8

print(TD0(P_act, pi_SWWS, R_allowed_etudiant, N_max, 0.001, v_0_etudiant, s_0, gamma, absorbing_state))
print(TD0(P_act, pi_SWWS, R_allowed_etudiant, N_max, 0.01, v_0_etudiant, s_0, gamma, absorbing_state))
print(TD0(P_act, pi_SWWS, R_allowed_etudiant, N_max, 0.05, v_0_etudiant, s_0, gamma, absorbing_state))
print(TD0(P_act, pi_SWWS, R_allowed_etudiant, N_max, 0.1, v_0_etudiant, s_0, gamma, absorbing_state))

[25.1507813  36.53401126 55.80289134 96.71231583  0.          0.
  0.        ]
[25.19949646 37.41649257 56.78341795 96.62563407  0.          0.
  0.        ]
[23.69717938 34.70393928 53.0037538  95.50653991  0.          0.
  0.        ]
[27.40142964 36.98902311 52.76481839 98.4024576   0.          0.
  0.        ]
